### 기능 통합

In [ ]:
# 답변 템플릿
reply_templates = {
    1: {
        "title": "자료 확인 완료",
        "body": """안녕하세요.

보내주신 자료 확인했습니다.
검토 후 필요한 사항이 있으면 다시 연락드리겠습니다.

감사합니다."""
    },
    2: {
        "title": "검토 후 회신 예정",
        "body": """안녕하세요.

메일 확인했습니다.
내용 검토 후 회신드리겠습니다.

감사합니다."""
    },
    3: {
        "title": "첨부파일 재전송 요청",
        "body": """안녕하세요.

메일 내용은 확인했으나 첨부파일이 확인되지 않습니다.
첨부파일을 다시 전달해주시면 확인하겠습니다.

감사합니다."""
    },
    4: {
        "title": "일정 참석 가능",
        "body": """안녕하세요.

안내해주신 일정 확인했습니다.
해당 일정에 참석 가능합니다.

감사합니다."""
    },
    5: {
        "title": "처리 완료",
        "body": """안녕하세요.

요청하신 내용 처리 완료했습니다.
확인 부탁드립니다.

감사합니다."""
    }
}

In [ ]:
import imaplib
import email
from email.header import decode_header
import os
import re
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.utils import parseaddr

IMAP_SERVER = "imap.gmail.com"
EMAIL = "hyojun000404@gmail.com"
PASSWORD = "pzxz wgpx lkwb uako"

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

# 첨부파일 저장 폴더
SAVE_DIR = "./downloads"
# downloads 폴더가 없으면 새로 생성
os.makedirs(SAVE_DIR, exist_ok=True)

# 전역 DataFrame
df = pd.DataFrame()

def main():
    """
    Regex + IMAP 업무 자동화 프로그램 메인 메뉴
    """

    global df

    while True:
        print("\n========== Regex + IMAP 업무 자동화 ==========")
        print("1. 최신 이메일 불러오기")
        print("2. 가져온 이메일 CSV 저장")
        print("3. 처리 우선순위 추천 보기")
        print("4. 메일 데이터 시각화")
        print("5. 자동 답변 보내기")
        print("6. 메일 검색/필터")
        print("7. 처리 상태 변경")
        print("0. 종료")
        print("=============================================")

        menu = input("실행할 기능 번호를 입력하세요: ")

        if menu == "1":
            count = int(input("가져올 최신 이메일 개수를 입력하세요: "))
            df = load_recent_emails(count)

            print(f"\n최신 이메일 {len(df)}개를 불러왔습니다.")
            display(df)

        elif menu == "2":
            save_mail_csv(df)

        elif menu == "3":
            show_priority_recommendation(df)

        elif menu == "4":
            show_mail_dashboard(df)

        elif menu == "5":
            reply_selected_mail(df)

        elif menu == "6":
            filter_mails(df)

        elif menu == "7":
            df = update_mail_status(df)
            
        elif menu == "0":
            print("프로그램을 종료합니다.")
            break

        else:
            print("잘못된 입력입니다. 다시 선택하세요.")

In [ ]:
# 메일을 가져오는 함수
def load_recent_emails(count):
    """
    사용자가 지정한 개수만큼 최신 메일을 가져와 DataFrame으로 만드는 함수
    """

    mail = imaplib.IMAP4_SSL(IMAP_SERVER)
    mail.login(EMAIL, PASSWORD)
    mail.select("inbox")

    result, data = mail.search(None, "ALL")
    mail_ids = data[0].split()

    latest_mail_ids = mail_ids[-count:]

    rows = []

    for mail_id in reversed(latest_mail_ids):
        result, data = mail.fetch(mail_id, "(RFC822)")
        raw_email = data[0][1]
        msg = email.message_from_bytes(raw_email)

        sender = decode_text(msg["From"])
        subject = decode_text(msg["Subject"])
        body = get_body(msg)

        # 제목과 본문을 합쳐서 분석 대상으로 사용
        full_text = subject + " " + body
        # 첨부파일 여부 확인
        attachment_yn = has_attachment(msg)
        # 메일 분류 판단
        category = classify_mail(full_text)
        # 처리상태 판단
        status = get_status(full_text, attachment_yn)
        # 우선순위, 마감일 판단
        priority,deadline =  analyze_priority_deadline(full_text)

        # 메일 1개를 DataFrame의 행 1개로 만들기 위해 딕셔너리로 저장
        rows.append({
            "보낸 사람": sender,
            "제목": subject,
            "분류": category,
            "우선순위": priority,
            "처리상태": status,
            "마감일": deadline,
            "첨부파일여부": attachment_yn
        })
    # IMAP 연결 종료
    mail.logout()

    result_df = pd.DataFrame(rows)
    return result_df

#  메일을 csv 파일로 저장하는 함수
def save_mail_csv(df, filename="mail_task_list.csv"):
    """
    현재 DataFrame을 CSV 파일로 저장하는 함수
    """
    if df.empty:
        print("메일 데이터가 없습니다. 먼저 메일을 불러오세요.")
        return
    filename = input("저장할 CSV 파일명을 입력하세요(default: mail_task_list.csv): ")

    if filename.strip() == "":
        filename = "mail_task_list.csv"
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"CSV 저장 완료: {filename}")

# 처리 우선순위 출력 함수
def show_priority_recommendation(df):
    """
    처리 우선순위 추천 결과 출력
    """

    if df.empty:
        print("메일 데이터가 없습니다. 먼저 메일을 불러오세요.")
        return

    result_df = df.copy()
    result_df["마감일"] = pd.to_datetime(result_df["마감일"], errors="coerce")

    #filtered_df = result_df[result_df["우선순위"].isin(["높음", "중간"])].copy()
    filtered_df = result_df[(result_df["우선순위"].isin(["높음", "중간"])) & (df["처리상태"] != "처리 완료")].copy() # 처리 완료된 메일은 우선 순위 추천에서 제거

    if filtered_df.empty:
        print("우선 처리할 메일이 없습니다.")
        return

    priority_order = {
        "높음": 0,
        "중간": 1
    }

    filtered_df["우선순위정렬"] = filtered_df["우선순위"].map(priority_order)

    # 우선순위, 마감일 순으로 정렬
    filtered_df = filtered_df.sort_values(
        by=["우선순위정렬", "마감일"],
        ascending=[True, True]
    )
    print("\n===== 처리 우선순위 추천 결과 =====")
    display(filtered_df)

# 메일 제목/보낸 사람 디코딩 함수
def decode_text(text):
    """
    메일 제목이나 보낸 사람 정보가 인코딩되어 있을 수 있으므로
    사람이 읽을 수 있는 문자열로 변환하는 함수
    """

    if text is None:
        return ""

    decoded_parts = decode_header(text)
    result = ""

    for part, encoding in decoded_parts:
        # part가 bytes 타입이면 디코딩 필요
        if isinstance(part, bytes):
            result += part.decode(encoding or "utf-8", errors="ignore")
        else:
            result += part

    return result

# 메일 본문 추출 함수
def get_body(msg):
    """
    메일 객체에서 본문 텍스트를 추출하는 함수
    첨부파일 부분은 제외하고 text/plain 본문만 가져옴
    """

    body = ""

    # 메일이 여러 파트로 구성된 경우
    # 예: 본문 + 첨부파일 + HTML 본문
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get("Content-Disposition"))

            # 일반 텍스트 본문이고, 첨부파일이 아닌 경우만 추출
            if content_type == "text/plain" and "attachment" not in content_disposition:
                payload = part.get_payload(decode=True)

                if payload:
                    body += payload.decode(
                        part.get_content_charset() or "utf-8",
                        errors="ignore"
                    )

    # 메일이 단일 본문으로만 구성된 경우
    else:
        payload = msg.get_payload(decode=True)

        if payload:
            body = payload.decode(
                msg.get_content_charset() or "utf-8",
                errors="ignore"
            )

    return body

# 첨부파일 여부 확인 함수
def has_attachment(msg):
    """
    메일에 첨부파일이 있는지 확인하는 함수
    첨부파일 이름이 하나라도 있으면 Y 반환
    없으면 N 반환
    """

    for part in msg.walk():
        filename = part.get_filename()

        if filename:
            return "Y"

    return "N"

# 메일 분류 함수
def classify_mail(text):
    """
    제목 + 본문 내용을 기준으로 메일을 분류하는 함수
    미리 정한 키워드가 포함되어 있으면 해당 분류로 지정
    """

    # 카테고리를 위에서부터 순서대로 검사 -> 처음 발견되는 키워드를 바로 반환
    category_keywords = {
        "회의/일정": ["회의", "미팅", "일정", "참석", "zoom", "meet"],
        "마감/제출": ["마감", "제출", "기한", "오늘까지", "내일까지", "보고서"],
        "결제/영수증": ["결제", "영수증", "청구서", "입금", "세금계산서"],
        "문의/요청": ["문의", "요청", "확인 부탁", "검토", "회신", "답변"],
        "광고/홍보": ["이벤트", "할인", "프로모션", "뉴스레터", "쿠폰"]
    }

    # 분류별 키워드를 하나씩 검사
    for category, keywords in category_keywords.items():
        for keyword in keywords:
            if keyword.lower() in text.lower():
                return category

    # 어떤 키워드에도 해당하지 않으면 기타로 분류
    return "기타"

# 처리상태 판단 함수
def get_status(text, attachment_yn):
    """
    메일 내용과 첨부파일 여부를 기준으로
    사용자가 어떤 처리를 해야 하는지 추천하는 함수
    """

    # 위에서부터 일치하는 키워드가 있으면 바로 반환
    # 답장이 필요해 보이는 메일
    if any(k in text for k in ["회신", "답변", "확인 부탁", "검토 요청"]):
        return "답장 필요"

    # 마감이나 제출이 필요한 메일
    if any(k in text for k in ["마감", "제출", "기한", "오늘까지", "내일까지"]):
        return "처리 필요"

    # 첨부파일이 있는 경우
    if attachment_yn == "Y":
        return "첨부 확인 필요"

    # 기본값
    return "미처리"

# 우선순위 + 마감일 동시 판단 함수
def analyze_priority_deadline(text):
    """
    메일 내용(text)을 기반으로
    우선순위와 마감일을 동시에 판단하는 함수

    return:
        priority, deadline
    """

    text_lower = text.lower()
    today = datetime.today()

    high_keywords = ["긴급", "asap", "오늘까지", "내일까지", "익일까지", "금일까지", "마감", "중요"]
    middle_keywords = ["확인 부탁", "검토", "회신", "요청", "문의"]

    deadline_keywords = ["마감", "제출", "기한", "까지", "due"]

    deadline = ""

    # 1. 마감일 추출
    # 마감 관련 키워드가 있을 때만 날짜를 마감일로 판단
    if any(k in text_lower for k in deadline_keywords):

        # 오늘 관련 표현
        if any(k in text for k in ["오늘까지", "금일까지", "당일까지"]):
            deadline = today.strftime("%Y-%m-%d")

        # 내일 관련 표현
        elif (
            any(k in text for k in ["내일까지", "익일까지"])
            or ("익일" in text and "까지" in text)
        ):
            deadline = (today + timedelta(days=1)).strftime("%Y-%m-%d")

        # 2026-04-30 또는 2026.04.30
        else:
            pattern1 = r"(\d{4})[.-](\d{1,2})[.-](\d{1,2})"
            match = re.search(pattern1, text)

            if match:
                y, m, d = match.groups()
                deadline = f"{y}-{int(m):02d}-{int(d):02d}"

            else:
                # 4월 30일
                pattern2 = r"(\d{1,2})월\s*(\d{1,2})일"
                match = re.search(pattern2, text)

                if match:
                    m, d = match.groups()
                    deadline = f"{today.year}-{int(m):02d}-{int(d):02d}"

    # 2. 우선순위 판단
    priority = "낮음" # 기본값

    # 마감일이 있으면 남은 일수 기준으로 판단
    if deadline:
        deadline_date = datetime.strptime(deadline, "%Y-%m-%d")
        remain_days = (deadline_date.date() - today.date()).days

        if remain_days <= 3:
            priority = "높음"
        elif remain_days <= 5:
            priority = "중간"
        else:
            priority = "낮음"

     # 키워드 적용
    if any(keyword in text_lower for keyword in high_keywords):
        priority = "높음"

    # 우선순위가 이미 높음이라면 중간으로 변하지 않음
    if any(keyword in text_lower for keyword in middle_keywords) and (priority != "높음"):
        priority = "중간"

    return priority, deadline

# 메일 대시보드 함수
def show_mail_dashboard(df):
    """
    메일 종합 분석 대시보드
    """
    if df.empty:
        print("시각화할 메일 데이터가 없습니다. 먼저 이메일을 불러오세요.")
        return
    # 데이터 전처리
    df = df.copy()

    df["마감일"] = pd.to_datetime(df["마감일"], errors="coerce")

    today = pd.Timestamp.today().normalize()
    df["남은일수"] = (df["마감일"] - today).dt.days

    # 스타일
    sns.set(style="darkgrid", palette='deep')
    #plt.style.use('ggplot')
    plt.rcParams['font.family'] = "Malgun Gothic"

    # subplot 생성
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("메일 업무 종합 분석 대시보드", fontsize=18)
    # 1. 처리상태별 메일 수
    sns.countplot(
        data=df,
        x="처리상태",
        order=df["처리상태"].value_counts().index,
        ax=axes[0, 0]
    )
    axes[0, 0].set_title("처리상태별 메일 수")
    axes[0, 0].set_xlabel("")
    axes[0, 0].set_ylabel("메일 수")
    axes[0, 0].tick_params(axis='x', rotation=30)

    # 2. 분류별 메일 수
    sns.countplot(
        data=df,
        x="분류",
        order=df["분류"].value_counts().index,
        ax=axes[0, 1]
    )
    axes[0, 1].set_title("분류별 메일 수")
    axes[0, 1].set_xlabel("")
    axes[0, 1].set_ylabel("메일 수")
    axes[0, 1].tick_params(axis='x', rotation=30)

    # 3. 우선순위별 메일 수
    sns.countplot(
        data=df,
        x="우선순위",
        order=["높음", "중간", "낮음"],
        ax=axes[1, 0]
    )
    axes[1, 0].set_title("우선순위별 메일 수")
    axes[1, 0].set_xlabel("")
    axes[1, 0].set_ylabel("메일 수")

   # 4. 마감일까지 남은 일수 (scatter)
    deadline_df = df[df["남은일수"].notna()]

    sns.stripplot(
        data=deadline_df,
        x="남은일수",
        hue="우선순위",
        palette={"높음": "red", "중간": "orange", "낮음": "green"},
        ax=axes[1, 1],
        jitter=True
    )

    # 레이아웃 정리
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# 답변 전송 함수
def send_reply(to_email, original_subject, template):
    """
    선택한 템플릿으로 답장 메일을 보내는 함수
    """

    reply_subject = "Re: " + original_subject
    reply_body = template["body"]

    msg = MIMEMultipart()
    msg["From"] = EMAIL
    msg["To"] = to_email
    msg["Subject"] = reply_subject

    msg.attach(MIMEText(reply_body, "plain", "utf-8"))

    with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
        server.starttls()
        server.login(EMAIL, PASSWORD)
        server.send_message(msg)

    print("답장 전송 완료:", to_email)

# 답변 메일, 템플릿 선택 함수
def reply_selected_mail(df):
    """
    우선 처리 항목(우선순위 높음/중간) 중에서 메일을 선택하고,
    템플릿을 골라 자동 답장하는 함수
    """
    if df.empty:
        print("답변할 메일 데이터가 없습니다. 먼저 이메일을 불러오세요.")
        return

    df = df.copy()
    df["마감일"] = pd.to_datetime(df["마감일"], errors="coerce")

    filtered_df = df[df["우선순위"].isin(["높음", "중간"])].copy()

    if filtered_df.empty:
        print("우선 처리할 메일이 없습니다.")
        return

    priority_order = {
        "높음": 0,
        "중간": 1
    }

    filtered_df["우선순위정렬"] = filtered_df["우선순위"].map(priority_order)

    filtered_df = filtered_df.sort_values(
        by=["우선순위정렬", "마감일"],
        ascending=[True, True]
    )

    print("\n===== 우선 처리 항목 =====")
    display(filtered_df)

    mail_index = int(input("\n답장할 메일의 index를 입력하세요: "))

    if mail_index not in filtered_df.index:
        print("잘못된 index입니다.")
        return

    selected_mail = filtered_df.loc[mail_index]

    sender_text = selected_mail["보낸 사람"]
    sender_email = parseaddr(sender_text)[1]
    original_subject = selected_mail["제목"]

    while True:
        print("\n===== 답변 템플릿 목록 =====")
        for num, template in reply_templates.items():
            print(f"{num}. {template['title']}")

        template_num = int(input("\n사용할 템플릿 번호를 입력하세요: "))

        if template_num not in reply_templates:
            print("잘못된 템플릿 번호입니다.")
            continue

        selected_template = reply_templates[template_num]

        print("\n===== 전송될 답장 내용 =====")
        print("받는 사람:", sender_email)
        print("제목:", "Re: " + original_subject)
        print("내용:")
        print(selected_template["body"])

        confirm = input("\n전송할까요? (y: 전송 / n: 취소 / r: 템플릿 다시 선택): ")

        if confirm.lower() == "y":
            send_reply(sender_email, original_subject, selected_template)
            break

        elif confirm.lower() == "n":
            print("전송을 취소했습니다.")
            break

        elif confirm.lower() == "r":
            print("템플릿을 다시 선택합니다.")
            continue

        else:
            print("잘못된 입력입니다. y, n, r 중 하나를 입력하세요.")

#### 메일 입력/필터 함수

In [ ]:
def filter_mails(df):
    """
    메일 검색/필터 기능
    """

    if df.empty:
        print("메일 데이터가 없습니다. 먼저 이메일을 불러오세요.")
        return

    temp_df = df.copy()
    temp_df["마감일"] = pd.to_datetime(temp_df["마감일"], errors="coerce")

    print("\n===== 메일 검색/필터 =====")
    print("1. 보낸 사람 검색")
    print("2. 제목 키워드 검색")
    print("3. 분류별 보기")
    print("4. 우선순위별 보기")
    print("5. 처리상태별 보기")
    print("6. 첨부파일 있는 메일만 보기")
    print("7. 마감일 있는 메일만 보기")
    print("0. 돌아가기")

    menu = input("필터 번호를 선택하세요: ")

    if menu == "1":
        keyword = input("보낸 사람 키워드 입력: ")
        result_df = temp_df[temp_df["보낸 사람"].str.contains(keyword, case=False, na=False)]

    elif menu == "2":
        keyword = input("제목 키워드 입력: ")
        result_df = temp_df[temp_df["제목"].str.contains(keyword, case=False, na=False)]

    elif menu == "3":
        category = input("분류 입력(예: 회의/일정, 마감/제출, 문의/요청): ")
        result_df = temp_df[temp_df["분류"] == category]

    elif menu == "4":
        priority = input("우선순위 입력(높음/중간/낮음): ")
        result_df = temp_df[temp_df["우선순위"] == priority]

    elif menu == "5":
        status = input("처리상태 입력(미처리/답장 필요/처리 필요/첨부 확인 필요/처리 완료): ")
        result_df = temp_df[temp_df["처리상태"] == status]

    elif menu == "6":
        result_df = temp_df[temp_df["첨부파일여부"] == "Y"]

    elif menu == "7":
        result_df = temp_df[temp_df["마감일"].notna()]

    elif menu == "0":
        return

    else:
        print("잘못된 입력입니다.")
        return

    print(f"\n검색 결과: {len(result_df)}개")
    display(result_df)

#### 처리상태 변경 함수

In [ ]:
def update_mail_status(df):
    """
    메일의 처리상태를 변경하는 함수
    """

    if df.empty:
        print("메일 데이터가 없습니다. 먼저 이메일을 불러오세요.")
        return df

    temp_df = df.copy()

    print("\n===== 처리 상태 변경 =====")

    # 보기 좋게 인덱스 재정렬
    temp_df = temp_df.reset_index(drop=True)

    display(temp_df)

    # 변경할 메일 선택
    try:
        idx = int(input("\n상태를 변경할 메일 번호(index)를 입력하세요: "))
    except:
        print("숫자를 입력해주세요.")
        return df

    if idx < 0 or idx >= len(temp_df):
        print("잘못된 번호입니다.")
        return df

    # 현재 상태 출력
    print(f"\n현재 상태: {temp_df.loc[idx, '처리상태']}")

    # 상태 선택
    print("\n변경할 상태를 선택하세요:")
    print("1. 미처리")
    print("2. 처리 필요")
    print("3. 답장 필요")
    print("4. 첨부 확인 필요")
    print("5. 처리 완료")

    choice = input("번호 입력: ")

    status_map = {
        "1": "미처리",
        "2": "처리 필요",
        "3": "답장 필요",
        "4": "첨부 확인 필요",
        "5": "처리 완료"
    }

    if choice not in status_map:
        print("잘못된 입력입니다.")
        return df

    new_status = status_map[choice]

    # 상태 변경
    temp_df.loc[idx, "처리상태"] = new_status

    print("\n===== 변경 완료 =====")
    display(temp_df.loc[[idx]])

    # 원본 df에도 반영하려면 index 맞춰야 함
    original_index = df.index[idx]
    df.loc[original_index, "처리상태"] = new_status

    return df

In [12]:
# 메인 함수 실행
main()


최신 이메일 20개를 불러왔습니다.


,보낸 사람,제목,분류,우선순위,처리상태,마감일,첨부파일여부
0,hehe7821 <noreply@github.com>,hehe7821 invited you to hehe7821/IMAPemailprj,기타,낮음,미처리,,N
1,정효준 <eric000404@naver.com>,서버 점검 사전 안내,기타,낮음,미처리,2026-05-08,N
2,정효준 <eric000404@naver.com>,교육 신청 안내,기타,낮음,미처리,2026-05-05,N
3,정효준 <eric000404@naver.com>,계정 정보 업데이트 안내,문의/요청,중간,답장 필요,2026-05-03,N
4,정효준 <eric000404@naver.com>,설문조사 참여 요청,마감/제출,중간,처리 필요,2026-05-02,N
5,정효준 <eric000404@naver.com>,프로젝트 진행 상황 확인 요청 (5월 8일까지),문의/요청,중간,답장 필요,2026-05-08,N
6,정효준 <eric000404@naver.com>,자료 검토 요청드립니다,문의/요청,중간,답장 필요,2026-05-03,Y
7,정효준 <eric000404@naver.com>,🎉 오늘만! 50% 할인 이벤트 🎉,광고/홍보,낮음,첨부 확인 필요,,Y
8,정효준 <eric000404@naver.com>,시스템 점검 안내,기타,낮음,미처리,,N
9,정효준 <eric000404@naver.com>,면접 일정 안내,회의/일정,중간,답장 필요,,N



========== Regex + IMAP 업무 자동화 ==========
1. 최신 이메일 불러오기
2. 가져온 이메일 CSV 저장
3. 처리 우선순위 추천 보기
4. 메일 데이터 시각화
5. 자동 답변 보내기
6. 메일 검색/필터
7. 처리 상태 변경
0. 종료

===== 처리 상태 변경 =====


,보낸 사람,제목,분류,우선순위,처리상태,마감일,첨부파일여부
0,hehe7821 <noreply@github.com>,hehe7821 invited you to hehe7821/IMAPemailprj,기타,낮음,미처리,,N
1,정효준 <eric000404@naver.com>,서버 점검 사전 안내,기타,낮음,미처리,2026-05-08,N
2,정효준 <eric000404@naver.com>,교육 신청 안내,기타,낮음,미처리,2026-05-05,N
3,정효준 <eric000404@naver.com>,계정 정보 업데이트 안내,문의/요청,중간,답장 필요,2026-05-03,N
4,정효준 <eric000404@naver.com>,설문조사 참여 요청,마감/제출,중간,처리 필요,2026-05-02,N
5,정효준 <eric000404@naver.com>,프로젝트 진행 상황 확인 요청 (5월 8일까지),문의/요청,중간,답장 필요,2026-05-08,N
6,정효준 <eric000404@naver.com>,자료 검토 요청드립니다,문의/요청,중간,답장 필요,2026-05-03,Y
7,정효준 <eric000404@naver.com>,🎉 오늘만! 50% 할인 이벤트 🎉,광고/홍보,낮음,첨부 확인 필요,,Y
8,정효준 <eric000404@naver.com>,시스템 점검 안내,기타,낮음,미처리,,N
9,정효준 <eric000404@naver.com>,면접 일정 안내,회의/일정,중간,답장 필요,,N



현재 상태: 첨부 확인 필요

변경할 상태를 선택하세요:
1. 미처리
2. 처리 필요
3. 답장 필요
4. 첨부 확인 필요
5. 처리 완료

===== 변경 완료 =====


...


========== Regex + IMAP 업무 자동화 ==========
1. 최신 이메일 불러오기
2. 가져온 이메일 CSV 저장
3. 처리 우선순위 추천 보기
4. 메일 데이터 시각화
5. 자동 답변 보내기
6. 메일 검색/필터
7. 처리 상태 변경
0. 종료

===== 메일 검색/필터 =====
1. 보낸 사람 검색
2. 제목 키워드 검색
3. 분류별 보기
4. 우선순위별 보기
5. 처리상태별 보기
6. 첨부파일 있는 메일만 보기
7. 마감일 있는 메일만 보기
0. 돌아가기

검색 결과: 8개


,보낸 사람,제목,분류,우선순위,처리상태,마감일,첨부파일여부
1,정효준 <eric000404@naver.com>,서버 점검 사전 안내,기타,낮음,미처리,2026-05-08,N
2,정효준 <eric000404@naver.com>,교육 신청 안내,기타,낮음,미처리,2026-05-05,N
3,정효준 <eric000404@naver.com>,계정 정보 업데이트 안내,문의/요청,중간,답장 필요,2026-05-03,N
4,정효준 <eric000404@naver.com>,설문조사 참여 요청,마감/제출,중간,처리 필요,2026-05-02,N
5,정효준 <eric000404@naver.com>,프로젝트 진행 상황 확인 요청 (5월 8일까지),문의/요청,중간,답장 필요,2026-05-08,N
6,정효준 <eric000404@naver.com>,자료 검토 요청드립니다,문의/요청,중간,답장 필요,2026-05-03,Y
10,정효준 <eric000404@naver.com>,[긴급] 오늘까지 보고 필요,마감/제출,높음,처리 필요,2026-04-28,N
16,정효준(***0***171) <hyojun@kau.kr>,[과제 제출 안내] 4월 30일까지 보고서 제출 바랍니다,마감/제출,높음,처리 필요,2026-04-30,Y



========== Regex + IMAP 업무 자동화 ==========
1. 최신 이메일 불러오기
2. 가져온 이메일 CSV 저장
3. 처리 우선순위 추천 보기
4. 메일 데이터 시각화
5. 자동 답변 보내기
6. 메일 검색/필터
7. 처리 상태 변경
0. 종료

===== 메일 검색/필터 =====
1. 보낸 사람 검색
2. 제목 키워드 검색
3. 분류별 보기
4. 우선순위별 보기
5. 처리상태별 보기
6. 첨부파일 있는 메일만 보기
7. 마감일 있는 메일만 보기
0. 돌아가기

검색 결과: 1개


,보낸 사람,제목,분류,우선순위,처리상태,마감일,첨부파일여부
7,정효준 <eric000404@naver.com>,🎉 오늘만! 50% 할인 이벤트 🎉,광고/홍보,낮음,처리 완료,NaT,Y



========== Regex + IMAP 업무 자동화 ==========
1. 최신 이메일 불러오기
2. 가져온 이메일 CSV 저장
3. 처리 우선순위 추천 보기
4. 메일 데이터 시각화
5. 자동 답변 보내기
6. 메일 검색/필터
7. 처리 상태 변경
0. 종료
프로그램을 종료합니다.
